# DE2 - Data Cleaning

## Load and Combine the Raw Data

In [1]:
import pandas as pd

sheets = pd.read_excel(
    "../data/raw/online_retail_II.xlsx",
    sheet_name=None
)

df = pd.concat(sheets.values(), ignore_index=True)

print("Starting rows:", len(df))

cleaning_log_rows = [("Starting dataset", 0, len(df))]


Starting rows: 1067371


## Remove Cancelled Invoices

### Why

Cancelled invoices were removed because they do not represent completed purchases and should not contribute to customer purchasing behavior.

In [2]:
before = len(df)

df = df[~df["Invoice"].astype(str).str.startswith("C")].copy()

removed = before - len(df)

print("Rows removed:", removed)
print("Rows remaining:", len(df))

cleaning_log_rows.append(("Remove cancelled invoices", removed, len(df)))


Rows removed: 19494
Rows remaining: 1047877


## Remove Non-Positive Quantities

### Why

Rows with non-positive quantities were removed because they represent returns, corrections, or invalid purchase quantities rather than completed positive purchases.

In [3]:
before = len(df)

df = df[df["Quantity"] > 0].copy()

removed = before - len(df)

print("Rows removed:", removed)
print("Rows remaining:", len(df))

cleaning_log_rows.append(("Remove non-positive quantities", removed, len(df)))


Rows removed: 3457
Rows remaining: 1044420


## Remove Non-Positive Prices

### Why

Rows with zero or negative prices were removed because they do not represent normal paid sales transactions.

In [4]:
before = len(df)

df = df[df["Price"] > 0].copy()

removed = before - len(df)

print("Rows removed:", removed)
print("Rows remaining:", len(df))

cleaning_log_rows.append(("Remove non-positive prices", removed, len(df)))


Rows removed: 2750
Rows remaining: 1041670


## Remove Missing Customer IDs

### Why

Rows without a Customer ID were removed because they cannot be assigned to a customer in the customer-product matrix. This means the recommender and best-seller baseline are based only on transactions from identified customers.

In [5]:
before = len(df)

df = df[df["Customer ID"].notna()].copy()

removed = before - len(df)

print("Rows removed:", removed)
print("Rows remaining:", len(df))

cleaning_log_rows.append(("Remove missing Customer IDs", removed, len(df)))


Rows removed:

 236121
Rows remaining: 805549


## Remove Non-Product Codes

### Why

Codes representing postage, carriage, discounts, adjustments, bank charges, or test transactions were removed because they are not actual retail products and should not appear as recommendable items.

`C2` (CARRIAGE), `DOT` (DOTCOM POSTAGE), `23444` (Next Day Carriage) and `23574` (PACKING CHARGE) are delivery or handling charges; `ADJUST` and `ADJUST2` are manual stock adjustments; `TEST001` and `TEST002` are test transactions. `23444` and `23574` use ordinary five-digit codes, so they are identifiable only from their descriptions.

`PADS` (PADS TO MATCH ALL CUSHIONS) and `SP1002` (KID'S CHALKBOARD/EASEL) were kept because their descriptions identify them as genuine products.

In [6]:
non_product_codes = [
    # service and bookkeeping codes
    "POST",
    "M",
    "D",
    "BANK CHARGES",
    "C2",
    "DOT",
    "ADJUST",
    "ADJUST2",
    # test transactions
    "TEST001",
    "TEST002",
    # charges that use ordinary-looking 5-digit codes
    "23444",
    "23574"
]

before = len(df)

# StockCode arrives from Excel as a mix of int and str: purely numeric codes
# load as integers, so compare as strings or numeric codes never match.
df = df[~df["StockCode"].astype(str).isin(non_product_codes)].copy()

removed = before - len(df)

print("Rows removed:", removed)
print("Rows remaining:", len(df))

cleaning_log_rows.append(("Remove non-product codes", removed, len(df)))


Rows removed:

 2991
Rows remaining: 802558


## Cleaning Log

In [7]:
cleaning_log = pd.DataFrame(
    cleaning_log_rows,
    columns=["Cleaning Rule", "Rows Removed", "Rows Remaining"]
)

cleaning_log


,Cleaning Rule,Rows Removed,Rows Remaining
0,Starting dataset,0,1067371
1,Remove cancelled invoices,19494,1047877
2,Remove non-positive quantities,3457,1044420
3,Remove non-positive prices,2750,1041670
4,Remove missing Customer IDs,236121,805549
5,Remove non-product codes,2991,802558


## Final Clean Dataset

In [8]:
print(df.shape)

(802558, 8)


## Save Cleaned Dataset


In [9]:
df.to_csv("../data/processed/cleaned_retail.csv", index=False)

print("Saved cleaned dataset.")

Saved cleaned dataset.
